# Uber Fare Prediction

End-to-end exploration for historical NYC Uber trips. Run `python fare_model.py` from the project directory to train the model used by the frontend, or run the cells below to review the data, compare models, and inspect errors.

**Scope:** model estimates based on historical fares; they are not live Uber quotes.

## 1. Load and inspect the data

The source includes an index-like column and a trip key that are not useful prediction features. The shared project helper parses timestamps and applies the documented NYC-area and fare/passenger filters.

In [ ]:
import pandas as pd
from fare_model import DATA_PATH, FEATURES, load_and_clean_data

raw = pd.read_csv(DATA_PATH)
print(f"Raw shape: {raw.shape}")
print("Missing values:\n", raw.isna().sum())
raw.head()

In [ ]:
df = load_and_clean_data()
print(f"Clean shape: {df.shape}")
print(f"Date range (UTC): {df.pickup_datetime.min()} to {df.pickup_datetime.max()}")
df[["fare_amount", "trip_distance_km", "passenger_count"]].describe().round(2)

## 2. Chronological holdout

Trips are sorted by pickup timestamp. The earliest 80% are used for training and the latest 20% for evaluation, reducing leakage from randomly mixing past and future trips.

In [ ]:
split_at = int(len(df) * 0.8)
train, test = df.iloc[:split_at], df.iloc[split_at:]
X_train, X_test = train[FEATURES], test[FEATURES]
y_train, y_test = train.fare_amount, test.fare_amount
print(f"Train: {len(train):,} rows through {train.pickup_datetime.max()}")
print(f"Test:  {len(test):,} rows from {test.pickup_datetime.min()}")

## 3. Compare a baseline and two distinct models

Linear regression provides an interpretable reference. Gradient boosting can capture nonlinear fare patterns. Scores on the chronological holdout are reported in dollars where applicable.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Median baseline": DummyRegressor(strategy="median"),
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Gradient boosting": GradientBoostingRegressor(
        n_estimators=120, learning_rate=0.06, max_depth=3,
        subsample=0.8, random_state=42
    ),
}
results, fitted = [], {}
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    fitted[name] = pred
    results.append({
        "Model": name,
        "MAE ($)": mean_absolute_error(y_test, pred),
        "RMSE ($)": mean_squared_error(y_test, pred) ** 0.5,
        "R-squared": r2_score(y_test, pred),
    })
results = pd.DataFrame(results).sort_values("MAE ($)")
results.round(3)

## 4. Inspect prediction errors

The best MAE model is plotted against actual fares and by residual. These plots help reveal systematic under/overprediction and error spread. The displayed metrics are specific to this dataset split and are not a guarantee of future performance.

In [ ]:
import matplotlib.pyplot as plt

best_name = results.iloc[0]["Model"]
pred = fitted[best_name]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, pred, alpha=0.25, s=8)
axes[0].plot([0, 200], [0, 200], "r--")
axes[0].set(xlabel="Actual fare ($)", ylabel="Predicted fare ($)", title=f"Actual vs predicted ? {best_name}")
residuals = y_test.to_numpy() - pred
axes[1].scatter(pred, residuals, alpha=0.25, s=8)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set(xlabel="Predicted fare ($)", ylabel="Actual ? predicted ($)", title="Residuals")
plt.tight_layout()

## 5. Train the frontend model

Run `python fare_model.py` to fit the production demo model on the first 80% of cleaned trips, evaluate on the final 20%, and save `fare_model.joblib` plus `model_metrics.json`. Launch the local web server with `python app.py`, then visit `http://127.0.0.1:8000`.